# NINA Output CSV for Spectra
## Adjust Celestial Coordinates for Assigned targets


In [1]:
!mkdir -p data_folder

# INITIALIZE CELESTIAL COORDINATES PROCESS

In [2]:
#pip install --upgrade astropy astroquery googlesearch-python wikipedia pandas


In [3]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.io import ascii
import numpy as np
import pandas as pd
# importing the module
import wikipedia as wiki
from IPython.display import Markdown as md

In [4]:
# ra_dec_offset_v5 and output files naming
notebook_name = "nina_targets_for_spectra"
notebook_version = "_v6"
data_folder_name = "data_folder"
output_csvfilename = f"{notebook_name}_{notebook_version}.csv"
print(f"{notebook_name} version is: {notebook_version}")
print(f"data_folder_name is: {data_folder_name}")
print(f"output_csvfilename is: {output_csvfilename}")

nina_targets_for_spectra version is: _v6
data_folder_name is: data_folder
output_csvfilename is: nina_targets_for_spectra__v6.csv


In [5]:
from astropy.coordinates import EarthLocation, AltAz, SkyCoord, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u

obs_tel = "BARO"
obs_loc = "San Diego"
obs_lat = 32.6 * u.deg  # for san diego
obs_lon = -116.3 * u.deg # for san diego
obs_hgt = 1131 * u.m # for BARO
safe_lim = 10 * u.deg # Account for BARO Telescop stops 
max_mag = 8 # max star magnitudes to consider
min_ra = 12 # min ra limit for BARO
max_ra = 18 # max ra limit for BARO

print(f"Observers Location is: {obs_loc}")
print(f"Observers Telescope is: {obs_tel}")
print(f"Observers Lattitude is: {obs_lat}")
print(f"Observers Longitude is: {obs_lon}")
print(f"Observers Height is: {obs_hgt}")
print(f"Safe Limit for {obs_tel} is: {safe_lim}")
print(f"Max Mag to Query is: {max_mag}")
print(f"Min RA to Query is: {min_ra}")
print(f"Max RA to Query is: {max_ra}")

# Define observer location
location = EarthLocation.from_geodetic(
    lat=obs_lat, lon=obs_lon, height=obs_hgt
)

# Define observation time
time = Time("2025-06-09 21:30:00")

# Define the celestial object's coordinates (e.g., RA and Dec)
sky_coord = SkyCoord(ra=10 * u.deg, dec=20 * u.deg)

# Create an AltAz frame
altaz_frame = AltAz(obstime=time, location=location)

# Transform the object's coordinates to AltAz
altaz_coord = sky_coord.transform_to(altaz_frame)

# Get the altitude and azimuth
altitude = altaz_coord.alt
azimuth = altaz_coord.az

print(f"Altitude: {altitude:.4f}")
print(f"Azimuth: {azimuth:.4f}")


# Define location and time
#location = EarthLocation(lat='32.7', lon='-116.33', height=0*u.m)
#obstime = Time.now()
#obstime = datetime.time(21, 0)

# AltAz frame for the observer
#altaz_frame = AltAz(obstime=obstime, location=location)

# Determine Declination range
min_dec = location.lat - 90*u.deg + safe_lim
max_dec = location.lat + 90*u.deg - safe_lim
print(f"Observable Declination range: {min_dec.to_string(unit=u.deg)} to {max_dec.to_string(unit=u.deg)}")

# Set global offset for spectra in frame to include zero-order
global_offset_arcmin = -3.5 # reduced from 3.5
print(f"Global offset RA & Dec arcmin by {global_offset_arcmin} * sin(camera rotation angle)")

Observers Location is: San Diego
Observers Telescope is: BARO
Observers Lattitude is: 32.6 deg
Observers Longitude is: -116.3 deg
Observers Height is: 1131.0 m
Safe Limit for BARO is: 10.0 deg
Max Mag to Query is: 8
Min RA to Query is: 12
Max RA to Query is: 18
Altitude: 7.1947 deg
Azimuth: 289.3402 deg
Observable Declination range: -47d24m00s to 112d36m00s
Global offset RA & Dec arcmin by -3.5 * sin(camera rotation angle)


In [6]:
obs_zen = 90 * u.deg - obs_lat
safe_min = obs_lat -90 * u.deg + safe_lim
safe_max = obs_lat +90 * u.deg - safe_lim
print(f'The Zenith at {obs_loc} is: {obs_zen:0.2f} deg')
print(f'Safe Declination limits at {obs_tel} are: {safe_min:0.2f} deg to {safe_max:0.2f} deg')

The Zenith at San Diego is: 57.40 deg deg
Safe Declination limits at BARO are: -47.40 deg deg to 112.60 deg deg


In [7]:
# set target default name
target_default_name = "HD"

In [8]:
def compute_exposure_time(mag: float) -> float:
    """
    Compute exposure time (in seconds) to reach 50,000 flux
    given the apparent magnitude, using the refit model
    (excluding La Superba).
    """
    a = 0.9325
    b = 1.0569
    c = -12.325
    target_flux = 50000

    log_flux = np.log(target_flux)
    log_exp = (log_flux + a * mag + c) / b
    return np.exp(log_exp)*2.5



In [9]:
def compute_adj_coord(original_ra_, original_dec_, offset_arcmin_, camera_rotation_deg_): 
    # --- Step 1A: Compute sky position angle for image "left" ---
    original_coord = SkyCoord(original_ra_, original_dec_)
    #original_coord_icrs = original_coord.transform_to('icrs')
    
    # --- Step 2: Compute sky position angle for image "left" ---
    sky_PA = (270 - camera_rotation_deg_) * u.deg
    
    print(f'\nsky_PA: {sky_PA}')
    
    # --- Step 3: Offset distance converted to tangent plane components ---
    offset_dist = offset_arcmin_ * u.arcmin
    
    print(f'\noffset_dist: {offset_dist}')
    
    dx = offset_dist * np.sin(sky_PA)
    dy = offset_dist * np.cos(sky_PA)
    
    print(f'\ndx: {dx} dy: {dy}')
          

    # --- Step 4: Define the offset frame centered on the original target ---
    offset_frame = SkyOffsetFrame(origin=original_coord)
    
    print(f'\noffset_frame = {offset_frame}')

    # --- Step 5: Create a coordinate in the offset frame and transform back ---
    offset_coord = SkyCoord(lon=dx, lat=dy, frame=offset_frame)
    new_coord_ = offset_coord.transform_to('icrs')
    
    print(f'\noffset_coord = {offset_coord}')
    print(f'\nnew_coord_ = {new_coord_}')
    
    return new_coord_



In [10]:
import requests
import time

def make_api_request(url, max_retries=5, initial_delay=2):
    retries = 0
    while retries < max_retries:
        try:
            response = requests.get(url)
            if response.status_code == 429:
                print(f"Received 429 error. Retrying in {initial_delay} seconds...")
                retry_after = response.headers.get('Retry-After')
                if retry_after:
                    delay = int(retry_after)
                else:
                    delay = initial_delay * (2 ** retries) # Exponential backoff
                time.sleep(delay)
                retries += 1
            elif response.status_code == 200:
                return response.json()
            else:
                print(f"Error: {response.status_code}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(initial_delay * (2 ** retries)) # Exponential backoff for network errors
            retries += 1
    print("Max retries exceeded. Request failed.")
    return None

# wikipedia Example usage
# to search
#query = f'https://en.wikipedia.org/wiki/{target_name} Wikipedia Astronomy'

#results = make_api_request(query)
#if results:
#    print("Data received:", results)


In [11]:
def googlesearchurl(tname):
    try:
        from googlesearch import search
    except ImportError:
        print("No module named 'google' found")
    #
    # to search
    query = f'{tname} Wikipedia Astronomy'
    
    for j in search(query, tld="co.us", num=10, stop=10, pause=2):
            print(j)
    #for url in search(query, num_results=10, lang="en", unique=True):
    #    print(url)
#
# example
#
#tname = "RR Lyrae"
#googlesearchurl(tname)

In [12]:
def wikisearchurl(tname):
    # Set language to English (optional)
    wiki.set_lang("en")
    
    # Search for a topic
    results = wiki.search(tname)
    print("Search Results:", results)
    
    # Get a summary of the first result
    summary = wiki.summary(results[0], sentences=1)
    print("\nSummary:", summary)
    
    # Get the full page object
    page = wiki.page(results[0])
    print("\nPage Title:", page.title)
    print("Page URL:", page.url)
#
# example
#
#tname = "RR Lyrae"
#wikisearchurl(tname)

In [13]:
def obtain_info_for_adhoc_star(tname, ara, adec):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(tname)
    original_coord = SkyCoord(ara, adec, frame='icrs', unit='deg')
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord)

In [14]:
def obtain_info_for_star(tname):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    original_coord = SkyCoord.from_name(tname)
    
    print(original_coord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(original_coord.ra, original_coord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {tname} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {original_coord.ra.hour:.4f}, {original_coord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {original_coord.ra.deg:.4f}, {original_coord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(original_coord.ra.deg-new_coord.ra.deg):.4f}, \
          {(original_coord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");wikisearchurl(tname);print("\n")

    return(new_coord, original_coord)

In [15]:
def obtain_adjcoord_for_planet(pname, pcoord):
    # --- Change User Inputs ---
    #offset_arcmin = -3.5                    # Offset distance (arcmin)
    offset_arcmin = global_offset_arcmin
    camera_rotation_deg = -21.9              # 1/12 of a full rotation = 30° clockwise
    
    # --- Step 1: Look up target coordinates ---
    #original_coord = SkyCoord.from_name(target_name)
    
    print(pcoord)
    
    # --- Step 1A-5 : Call function to calculate adjusted coordinates ---
    new_coord = compute_adj_coord(pcoord.ra, pcoord.dec, offset_arcmin, camera_rotation_deg)
    
    # --- Step 6: Report result ---
    print(f"Using SkyOffsetFrame for Star {target_name} ")
    print(f"User inputs: offset_arcmin = {offset_arcmin}, camera_rotation_deg = {camera_rotation_deg}")
    print(f"\nOriginal RA(Hr)/Dec: {pcoord.ra.hour:.4f}, {pcoord.dec.deg:.4f}")
    print(f"Original RA(Deg)/Dec: {pcoord.ra.deg:.4f}, {pcoord.dec.deg:.4f}")
    print(f"New  RA(Hr)/Dec:  {new_coord.ra.hour:.4f}, {new_coord.dec.deg:.4f}")
    print(f"New  RA(Deg)/Dec:  {new_coord.ra.deg:.4f}, {new_coord.dec.deg:.4f}")
    print(f"Delta RA/DEC(min): {(pcoord.ra.deg-new_coord.ra.deg):.4f}, \
          {(pcoord.dec.deg-new_coord.dec.deg):.4f}")
    
    #print("\n");googlesearchurl(target_name);print("\n")

    return(new_coord, orig_coord)

In [16]:
# ---  CREATE A DATAFRAME
column_names = ["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"] 
df = pd.DataFrame(columns=column_names)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)  # Or a large integer like 9999
ridx = 0

# CELESTIAL COORDINATES FOR NEW STAR 3C273

In [17]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "3C273"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star 3C273

In [18]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (187.27791594, 2.05238823)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (187.27791594, 2.05238823)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (187.27791594, 2.05238823)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (187.33207373, 2.03062969)>
Using SkyOffsetFrame for Star 3C273 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.4852, 2.0524
Original RA(Deg)/Dec: 187.2779, 2.0524
New  RA(Hr)/Dec:  12.4888, 2.0306
New  RA(Deg)/Dec:  187.3321, 2.0306
Delta RA/DEC(min): -0.0542,           0.0218


### Update Properties for Star based on wiki query results above 

In [19]:
star_magnitude = 12.9;  target_alt_name = "Quasar_3C273" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 12.9: 52772.22 s


In [20]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~","rahrdec","rahrdegdec","decdegdec"]])

                      Name1*        Name2*   RA2000*  D2000* Pmag~      Exp~  \
0  3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321  2.0306  12.9  52772.22   

   rahrdec rahrdegdec decdegdec  
0  12.4852   187.2779    2.0524  


# OUTPUT CONSOLIDATED CSV FILE FOR BARO

In [21]:
df.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(df[["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"]])

                      Name1*        Name2*   RA2000*  D2000* Pmag~      Exp~  \
0  3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321  2.0306  12.9  52772.22   

  Note1 Note2 NExp~ GetRef Temp  rahrdec rahrdegdec decdegdec  
0    NA    NA     1      0       12.4852   187.2779    2.0524  


<div class="alert alert-danger"><strong>STOP HERE AS NEEDED FOR CONCISE CSV TARGETS</strong></div>

# CELESTIAL COORDINATES FOR NEW STAR HR 5422

In [26]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "HR 5422"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star HR 5422

In [27]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (217.45696076, 31.79118853)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (217.45696076, 31.79118853)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (217.45696076, 31.79118853)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (217.52062277, 31.76941507)>
Using SkyOffsetFrame for Star HR 5422 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.4971, 31.7912
Original RA(Deg)/Dec: 217.4570, 31.7912
New  RA(Hr)/Dec:  14.5014, 31.7694
New  RA(Deg)/Dec:  217.5206, 31.7694
Delta RA/DEC(min): -0.0637,           0.0218


### Update Properties for Star based on wiki query results above 

In [28]:
star_magnitude = 6.05;  target_alt_name = "HD 127304" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0V'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 6.05: 125.21 s


In [29]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                      Name1*        Name2*   RA2000*   D2000* Pmag~      Exp~
0  3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306  12.9  52772.22
1   HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708  7.73    551.29
2    HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694  6.05    125.21


# CELESTIAL COORDINATES FOR NEW STAR Alphecca

In [30]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alphecca"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alphecca

In [31]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (233.67195203, 26.714685)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (233.67195203, 26.714685)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (233.67195203, 26.714685)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (233.73253201, 26.69291452)>
Using SkyOffsetFrame for Star Alphecca 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 15.5781, 26.7147
Original RA(Deg)/Dec: 233.6720, 26.7147
New  RA(Hr)/Dec:  15.5822, 26.6929
New  RA(Deg)/Dec:  233.7325, 26.6929
Delta RA/DEC(min): -0.0606,           0.0218


### Update Properties for Star based on wiki query results above 

In [32]:
star_magnitude = 2.22;  target_alt_name = "Alphecca" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0V'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.22: 4.27 s


In [33]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                      Name1*        Name2*   RA2000*   D2000* Pmag~      Exp~
0  3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306  12.9  52772.22
1   HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708  7.73    551.29
2    HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694  6.05    125.21
3  Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929  2.22      4.27


# CELESTIAL COORDINATES FOR NEW STAR Rasalhague

In [34]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Rasalhague"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Rasalhague

In [35]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (263.73362272, 12.56003739)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (263.73362272, 12.56003739)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (263.73362272, 12.56003739)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (263.78906882, 12.53827408)>
Using SkyOffsetFrame for Star Rasalhague 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 17.5822, 12.5600
Original RA(Deg)/Dec: 263.7336, 12.5600
New  RA(Hr)/Dec:  17.5859, 12.5383
New  RA(Deg)/Dec:  263.7891, 12.5383
Delta RA/DEC(min): -0.0554,           0.0218


### Update Properties for Star based on wiki query results above 

In [36]:
star_magnitude = 2.08;  target_alt_name = "Rasalhague" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'AVIII'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.08: 3.77 s


In [37]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                            Name1*        Name2*   RA2000*   D2000* Pmag~  \
0        3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306  12.9   
1         HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708  7.73   
2          HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694  6.05   
3        Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929  2.22   
4  Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383  2.08   

       Exp~  
0  52772.22  
1    551.29  
2    125.21  
3      4.27  
4      3.77  


In [38]:
df.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(df[["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"]])

                            Name1*        Name2*   RA2000*   D2000* Pmag~  \
0        3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306  12.9   
1         HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708  7.73   
2          HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694  6.05   
3        Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929  2.22   
4  Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383  2.08   

       Exp~ Note1 Note2 NExp~ GetRef Temp  rahrdec rahrdegdec decdegdec  
0  52772.22    NA    NA     1      0       12.4852   187.2779    2.0524  
1    551.29    NA    NA     1      0        6.4747    97.1206   34.4925  
2    125.21    NA    NA     1      0       14.4971   217.4570   31.7912  
3      4.27    NA    NA     1      0       15.5781   233.6720   26.7147  
4      3.77    NA    NA     1      0       17.5822   263.7336   12.5600  


<div class="alert alert-danger"><strong>STOP HERE AS NEEDED FOR CONCISE CSV TARGETS</strong></div>

# CELESTIAL COORDINATES FOR NEW STAR ALTAIR

In [39]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Altair"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Altair

In [40]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (297.6958273, 8.8683212)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (297.6958273, 8.8683212)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (297.6958273, 8.8683212)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (297.7506027, 8.84655959)>
Using SkyOffsetFrame for Star Altair 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.8464, 8.8683
Original RA(Deg)/Dec: 297.6958, 8.8683
New  RA(Hr)/Dec:  19.8500, 8.8466
New  RA(Deg)/Dec:  297.7506, 8.8466
Delta RA/DEC(min): -0.0548,           0.0218


### Update Properties for Star based on wiki query results above 

In [41]:
star_magnitude = 0.76;  target_alt_name = "HD 187642" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 0.76: 1.18 s


In [42]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                            Name1*        Name2*   RA2000*   D2000* Pmag~  \
0        3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306  12.9   
1         HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708  7.73   
2          HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694  6.05   
3        Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929  2.22   
4  Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383  2.08   
5          Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466  0.76   

       Exp~  
0  52772.22  
1    551.29  
2    125.21  
3      4.27  
4      3.77  
5      1.18  


# CELESTIAL COORDINATES FOR NEW STAR Vega

In [43]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Vega"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Vega

In [44]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (279.23473479, 38.78368896)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (279.23473479, 38.78368896)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (279.23473479, 38.78368896)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (279.30414611, 38.7619108)>
Using SkyOffsetFrame for Star Vega 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.6156, 38.7837
Original RA(Deg)/Dec: 279.2347, 38.7837
New  RA(Hr)/Dec:  18.6203, 38.7619
New  RA(Deg)/Dec:  279.3041, 38.7619
Delta RA/DEC(min): -0.0694,           0.0218


### Update Properties for Star based on wiki query results above 

In [45]:
star_magnitude = 0.026;  target_alt_name = "HD 172167" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 0.026: 0.62 s


In [46]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                            Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0        3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1         HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2          HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3        Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4  Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5          Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6            Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   

       Exp~  
0  52772.22  
1    551.29  
2    125.21  
3      4.27  
4      3.77  
5      1.18  
6      0.62  


# CELESTIAL COORDINATES FOR NEW STAR Dubhe

In [47]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Dubhe"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Dubhe

In [48]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (165.93196467, 61.75103469)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (165.93196467, 61.75103469)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (165.93196467, 61.75103469)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (166.04623695, 61.72922952)>
Using SkyOffsetFrame for Star Dubhe 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.0621, 61.7510
Original RA(Deg)/Dec: 165.9320, 61.7510
New  RA(Hr)/Dec:  11.0697, 61.7292
New  RA(Deg)/Dec:  166.0462, 61.7292
Delta RA/DEC(min): -0.1143,           0.0218


### Update Properties for Star based on wiki query results above 

In [49]:
star_magnitude = 1.79;  target_alt_name = "HD 95689" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.79: 2.92 s


In [50]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                            Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0        3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1         HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2          HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3        Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4  Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5          Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6            Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7            Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   

       Exp~  
0  52772.22  
1    551.29  
2    125.21  
3      4.27  
4      3.77  
5      1.18  
6      0.62  
7      2.92  


# CELESTIAL COORDINATES FOR NEW STAR Scheat

In [51]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Scheat"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Scheat

In [52]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (345.94357274, 28.08278712)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (345.94357274, 28.08278712)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (345.94357274, 28.08278712)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (346.00490648, 28.06101587)>
Using SkyOffsetFrame for Star Scheat 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 23.0629, 28.0828
Original RA(Deg)/Dec: 345.9436, 28.0828
New  RA(Hr)/Dec:  23.0670, 28.0610
New  RA(Deg)/Dec:  346.0049, 28.0610
Delta RA/DEC(min): -0.0613,           0.0218


### Update Properties for Star based on wiki query results above 

In [53]:
star_magnitude = 2.42;  target_alt_name = "HD 217906" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.42: 5.09 s


In [54]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                            Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0        3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1         HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2          HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3        Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4  Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5          Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6            Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7            Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8          Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   

       Exp~  
0  52772.22  
1    551.29  
2    125.21  
3      4.27  
4      3.77  
5      1.18  
6      0.62  
7      2.92  
8      5.09  


# CELESTIAL COORDINATES FOR NEW STAR Mizar

In [55]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Mizar"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Mizar

In [56]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (200.98141867, 54.92535197)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (200.98141867, 54.92535197)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (200.98141867, 54.92535197)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (201.07555446, 54.90355796)>
Using SkyOffsetFrame for Star Mizar 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 13.3988, 54.9254
Original RA(Deg)/Dec: 200.9814, 54.9254
New  RA(Hr)/Dec:  13.4050, 54.9036
New  RA(Deg)/Dec:  201.0756, 54.9036
Delta RA/DEC(min): -0.0941,           0.0218


### Update Properties for Star based on wiki query results above 

In [57]:
star_magnitude = 2.04;  target_alt_name = "HD 116656" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.04: 3.64 s


In [58]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                            Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0        3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1         HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2          HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3        Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4  Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5          Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6            Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7            Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8          Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9           Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   

       Exp~  
0  52772.22  
1    551.29  
2    125.21  
3      4.27  
4      3.77  
5      1.18  
6      0.62  
7      2.92  
8      5.09  
9

# CELESTIAL COORDINATES FOR NEW STAR Alcor

In [59]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alcor"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alcor

In [60]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (201.30640764, 54.98795966)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (201.30640764, 54.98795966)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (201.30640764, 54.98795966)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (201.4006901, 54.96616557)>
Using SkyOffsetFrame for Star Alcor 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 13.4204, 54.9880
Original RA(Deg)/Dec: 201.3064, 54.9880
New  RA(Hr)/Dec:  13.4267, 54.9662
New  RA(Deg)/Dec:  201.4007, 54.9662
Delta RA/DEC(min): -0.0943,           0.0218


### Update Properties for Star based on wiki query results above 

In [61]:
star_magnitude = 3.88;  target_alt_name = "HD 116657" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'MK'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.88: 18.46 s


In [62]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   

        Exp~  
0   52772.22  
1     551.29  
2     

# CELESTIAL COORDINATES FOR NEW STAR R Lyr

In [63]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "R Lyr"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star R Lyr

In [64]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (283.83375974, 43.94608958)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (283.83375974, 43.94608958)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (283.83375974, 43.94608958)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (283.90890485, 43.92430733)>
Using SkyOffsetFrame for Star R Lyr 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.9223, 43.9461
Original RA(Deg)/Dec: 283.8338, 43.9461
New  RA(Hr)/Dec:  18.9273, 43.9243
New  RA(Deg)/Dec:  283.9089, 43.9243
Delta RA/DEC(min): -0.0751,           0.0218


### Update Properties for Star based on wiki query results above 

In [65]:
star_magnitude = 3.9;  target_alt_name = "HD 175865" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.9: 18.79 s


In [66]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Alpheratz

In [67]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alpheratz"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alpheratz

In [68]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (2.09691619, 29.09043112)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (2.09691619, 29.09043112)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (2.09691619, 29.09043112)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (2.15884001, 29.06865928)>
Using SkyOffsetFrame for Star Alpheratz 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.1398, 29.0904
Original RA(Deg)/Dec: 2.0969, 29.0904
New  RA(Hr)/Dec:  0.1439, 29.0687
New  RA(Deg)/Dec:  2.1588, 29.0687
Delta RA/DEC(min): -0.0619,           0.0218


### Update Properties for Star based on wiki query results above 

In [69]:
star_magnitude = 2.06;  target_alt_name = "HD 358" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8_A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.06: 3.70 s


In [70]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Albireo A

In [71]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Albireo"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Albireo

In [72]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (292.68031501, 27.95967363)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (292.74157872, 27.93790244)>
Using SkyOffsetFrame for Star Albireo 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.5120, 27.9597
Original RA(Deg)/Dec: 292.6803, 27.9597
New  RA(Hr)/Dec:  19.5161, 27.9379
New  RA(Deg)/Dec:  292.7416, 27.9379
Delta RA/DEC(min): -0.0613,           0.0218


### Update Properties for Star based on wiki query results above 

In [73]:
star_magnitude = 3.21;  target_alt_name = "HD 183912" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.21: 10.22 s


In [74]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Albireo B

In [75]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Albireo"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Albireo

In [76]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (292.68031501, 27.95967363)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (292.68031501, 27.95967363)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (292.74157872, 27.93790244)>
Using SkyOffsetFrame for Star Albireo 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 19.5120, 27.9597
Original RA(Deg)/Dec: 292.6803, 27.9597
New  RA(Hr)/Dec:  19.5161, 27.9379
New  RA(Deg)/Dec:  292.7416, 27.9379
Delta RA/DEC(min): -0.0613,           0.0218


### Update Properties for Star based on wiki query results above 

In [77]:
star_magnitude = 5.11;  target_alt_name = "HD 183913" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B8'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 5.11: 54.63 s


In [78]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Denebola

In [79]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Denebola"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Denebola

In [80]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (177.26490976, 14.57205807)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (177.26490976, 14.57205807)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (177.26490976, 14.57205807)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (177.32082694, 14.5502938)>
Using SkyOffsetFrame for Star Denebola 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.8177, 14.5721
Original RA(Deg)/Dec: 177.2649, 14.5721
New  RA(Hr)/Dec:  11.8214, 14.5503
New  RA(Deg)/Dec:  177.3208, 14.5503
Delta RA/DEC(min): -0.0559,           0.0218


### Update Properties for Star based on wiki query results above 

In [81]:
star_magnitude = 2.14;  target_alt_name = "HD 102647" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A3'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.14: 3.98 s


In [82]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Zosma

In [83]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zosma"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Zosma

In [84]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (168.52708927, 20.52371814)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (168.52708927, 20.52371814)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (168.58487305, 20.50195095)>
Using SkyOffsetFrame for Star Zosma 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 11.2351, 20.5237
Original RA(Deg)/Dec: 168.5271, 20.5237
New  RA(Hr)/Dec:  11.2390, 20.5020
New  RA(Deg)/Dec:  168.5849, 20.5020
Delta RA/DEC(min): -0.0578,           0.0218


### Update Properties for Star based on wiki query results above 

In [85]:
star_magnitude = 2.56;  target_alt_name = "HD 97603" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.56: 5.76 s


In [86]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Alioth

In [87]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alioth"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alioth

In [88]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (193.50728997, 55.95982296)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.50728997, 55.95982296)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.50728997, 55.95982296)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (193.60392419, 55.93802752)>
Using SkyOffsetFrame for Star Alioth 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.9005, 55.9598
Original RA(Deg)/Dec: 193.5073, 55.9598
New  RA(Hr)/Dec:  12.9069, 55.9380
New  RA(Deg)/Dec:  193.6039, 55.9380
Delta RA/DEC(min): -0.0966,           0.0218


### Update Properties for Star based on wiki query results above 

In [89]:
star_magnitude = 1.77;  target_alt_name = "HD 112185" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.77: 2.87 s


In [90]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Minelauva

In [91]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Minelauva"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Minelauva

In [92]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (193.90086927, 3.3974689)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.90086927, 3.3974689)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (193.90086927, 3.3974689)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (193.95508712, 3.37570977)>
Using SkyOffsetFrame for Star Minelauva 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 12.9267, 3.3975
Original RA(Deg)/Dec: 193.9009, 3.3975
New  RA(Hr)/Dec:  12.9303, 3.3757
New  RA(Deg)/Dec:  193.9551, 3.3757
Delta RA/DEC(min): -0.0542,           0.0218


### Update Properties for Star based on wiki query results above 

In [93]:
star_magnitude = 3.32;  target_alt_name = "HD 112300" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M3'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.32: 11.26 s


In [94]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Arcturus

In [95]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Arcturus"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Arcturus

In [96]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (213.9153003, 19.18240916)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (213.9153003, 19.18240916)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (213.9153003, 19.18240916)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (213.97259826, 19.16064265)>
Using SkyOffsetFrame for Star Arcturus 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.2610, 19.1824
Original RA(Deg)/Dec: 213.9153, 19.1824
New  RA(Hr)/Dec:  14.2648, 19.1606
New  RA(Deg)/Dec:  213.9726, 19.1606
Delta RA/DEC(min): -0.0573,           0.0218


### Update Properties for Star based on wiki query results above 

In [97]:
star_magnitude = -0.05;  target_alt_name = "HD 124897" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag -0.05: 0.58 s


In [98]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR P Cyg

In [99]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "P Cyg"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star P Cyg

In [100]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (304.44667489, 38.03293031)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (304.44667489, 38.03293031)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (304.44667489, 38.03293031)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (304.5153694, 38.0111527)>
Using SkyOffsetFrame for Star P Cyg 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 20.2964, 38.0329
Original RA(Deg)/Dec: 304.4467, 38.0329
New  RA(Hr)/Dec:  20.3010, 38.0112
New  RA(Deg)/Dec:  304.5154, 38.0112
Delta RA/DEC(min): -0.0687,           0.0218


### Update Properties for Star based on wiki query results above 

In [101]:
star_magnitude = 4.82;  target_alt_name = "HD 193237" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B1'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.82: 42.30 s


In [102]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Polaris

In [103]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Polaris"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Polaris

In [104]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (37.95456067, 89.26410897)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (37.95456067, 89.26410897)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (37.95456067, 89.26410897)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (42.04074999, 89.24042071)>
Using SkyOffsetFrame for Star Polaris 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 2.5303, 89.2641
Original RA(Deg)/Dec: 37.9546, 89.2641
New  RA(Hr)/Dec:  2.8027, 89.2404
New  RA(Deg)/Dec:  42.0407, 89.2404
Delta RA/DEC(min): -4.0862,           0.0237


### Update Properties for Star based on wiki query results above 

In [105]:
star_magnitude = 1.98;  target_alt_name = "HD 8890" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.98: 3.45 s


In [106]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Zet1 Lyr

In [107]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zet1 Lyr"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Zet1 Lyr

In [108]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (281.19315451, 37.60512165)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.19315451, 37.60512165)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.19315451, 37.60512165)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (281.26145235, 37.58334435)>
Using SkyOffsetFrame for Star Zet1 Lyr 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.7462, 37.6051
Original RA(Deg)/Dec: 281.1932, 37.6051
New  RA(Hr)/Dec:  18.7508, 37.5833
New  RA(Deg)/Dec:  281.2615, 37.5833
Delta RA/DEC(min): -0.0683,           0.0218


### Update Properties for Star based on wiki query results above 

In [109]:
star_magnitude = 4.37;  target_alt_name = "HD 173648" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'kA5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.37: 28.44 s


In [110]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Zet2 Lyr

In [111]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Zet2 Lyr"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Zet2 Lyr

In [112]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (281.20082994, 37.5945996)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.20082994, 37.5945996)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (281.20082994, 37.5945996)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (281.26911813, 37.5728223)>
Using SkyOffsetFrame for Star Zet2 Lyr 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 18.7467, 37.5946
Original RA(Deg)/Dec: 281.2008, 37.5946
New  RA(Hr)/Dec:  18.7513, 37.5728
New  RA(Deg)/Dec:  281.2691, 37.5728
Delta RA/DEC(min): -0.0683,           0.0218


### Update Properties for Star based on wiki query results above 

In [113]:
star_magnitude = 5.74;  target_alt_name = "HD 173649" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 5.74: 95.25 s


In [114]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Kochab

In [115]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Kochab"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Kochab

In [116]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (222.6763575, 74.15550394)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (222.6763575, 74.15550394)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (222.6763575, 74.15550394)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (222.87432758, 74.13365636)>
Using SkyOffsetFrame for Star Kochab 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.8451, 74.1555
Original RA(Deg)/Dec: 222.6764, 74.1555
New  RA(Hr)/Dec:  14.8583, 74.1337
New  RA(Deg)/Dec:  222.8743, 74.1337
Delta RA/DEC(min): -0.1980,           0.0218


### Update Properties for Star based on wiki query results above 

In [117]:
star_magnitude = 2.08;  target_alt_name = "HD 131873" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.08: 3.77 s


In [118]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*   D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321   2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863  34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206  31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325  26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891  12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506   8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041  38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462  61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049  28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756  54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007  54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     HD 175865  2

# CELESTIAL COORDINATES FOR NEW STAR Dschubba

In [119]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Dschubba"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Dschubba

In [120]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (240.08335535, -22.62170643)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (240.08335535, -22.62170643)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (240.08335535, -22.62170643)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (240.1419995, -22.64345339)>
Using SkyOffsetFrame for Star Dschubba 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 16.0056, -22.6217
Original RA(Deg)/Dec: 240.0834, -22.6217
New  RA(Hr)/Dec:  16.0095, -22.6435
New  RA(Deg)/Dec:  240.1420, -22.6435
Delta RA/DEC(min): -0.0586,           0.0217


### Update Properties for Star based on wiki query results above 

In [121]:
star_magnitude = 1.59;  target_alt_name = "HD 143275" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'B0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 1.59: 2.45 s


In [122]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR Enif

In [123]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Enif"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Enif

In [124]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (326.04648391, 9.87500865)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (326.04648391, 9.87500865)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (326.04648391, 9.87500865)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (326.10141801, 9.85324658)>
Using SkyOffsetFrame for Star Enif 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 21.7364, 9.8750
Original RA(Deg)/Dec: 326.0465, 9.8750
New  RA(Hr)/Dec:  21.7401, 9.8532
New  RA(Deg)/Dec:  326.1014, 9.8532
Delta RA/DEC(min): -0.0549,           0.0218


### Update Properties for Star based on wiki query results above 

In [125]:
star_magnitude = 2.37;  target_alt_name = "HD 206778" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.37: 4.87 s


In [126]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR Alphecca

In [127]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Alphecca"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Alphecca

In [128]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (233.67195203, 26.714685)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (233.67195203, 26.714685)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (233.67195203, 26.714685)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (233.73253201, 26.69291452)>
Using SkyOffsetFrame for Star Alphecca 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 15.5781, 26.7147
Original RA(Deg)/Dec: 233.6720, 26.7147
New  RA(Hr)/Dec:  15.5822, 26.6929
New  RA(Deg)/Dec:  233.7325, 26.6929
Delta RA/DEC(min): -0.0606,           0.0218


### Update Properties for Star based on wiki query results above 

In [129]:
star_magnitude = 2.24;  target_alt_name = "HD 139006" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.24: 4.34 s


In [130]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR Eltanin

In [131]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Eltanin"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Eltanin

In [132]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (269.15154118, 51.48889562)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (269.15154118, 51.48889562)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (269.15154118, 51.48889562)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (269.23842229, 51.46710589)>
Using SkyOffsetFrame for Star Eltanin 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 17.9434, 51.4889
Original RA(Deg)/Dec: 269.1515, 51.4889
New  RA(Hr)/Dec:  17.9492, 51.4671
New  RA(Deg)/Dec:  269.2384, 51.4671
Delta RA/DEC(min): -0.0869,           0.0218


### Update Properties for Star based on wiki query results above 

In [133]:
star_magnitude = 2.23;  target_alt_name = "HD 164058" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'K5'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 2.23: 4.30 s


In [134]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR Thuban

In [135]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Thuban"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Thuban

In [136]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (211.09732332, 64.37586962)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (211.09732332, 64.37586962)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (211.09732332, 64.37586962)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (211.22237582, 64.35405874)>
Using SkyOffsetFrame for Star Thuban 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 14.0732, 64.3759
Original RA(Deg)/Dec: 211.0973, 64.3759
New  RA(Hr)/Dec:  14.0815, 64.3541
New  RA(Deg)/Dec:  211.2224, 64.3541
Delta RA/DEC(min): -0.1251,           0.0218


### Update Properties for Star based on wiki query results above 

In [137]:
star_magnitude = 3.67;  target_alt_name = "HD 123299" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.67: 15.33 s


In [138]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR h Uma

In [139]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "h Uma"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star h Uma

In [140]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (142.88211696, 63.06185995)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (142.88211696, 63.06185995)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (142.88211696, 63.06185995)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (143.00149871, 63.04005206)>
Using SkyOffsetFrame for Star h Uma 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 9.5255, 63.0619
Original RA(Deg)/Dec: 142.8821, 63.0619
New  RA(Hr)/Dec:  9.5334, 63.0401
New  RA(Deg)/Dec:  143.0015, 63.0401
Delta RA/DEC(min): -0.1194,           0.0218


### Update Properties for Star based on wiki query results above 

In [141]:
star_magnitude = 3.65;  target_alt_name = "HD 81937" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'F0'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 3.65: 15.07 s


In [142]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR Theta Cep

In [143]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Theta Cep"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Theta Cep

In [144]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (307.39543861, 62.99411033)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (307.39543861, 62.99411033)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (307.39543861, 62.99411033)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (307.51454356, 62.97230259)>
Using SkyOffsetFrame for Star Theta Cep 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 20.4930, 62.9941
Original RA(Deg)/Dec: 307.3954, 62.9941
New  RA(Hr)/Dec:  20.5010, 62.9723
New  RA(Deg)/Dec:  307.5145, 62.9723
Delta RA/DEC(min): -0.1191,           0.0218


### Update Properties for Star based on wiki query results above 

In [145]:
star_magnitude = 4.22;  target_alt_name = "HD 195725" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'A7'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.22: 24.91 s


In [146]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR VZ Cam

In [147]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "VZ Cam"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star VZ Cam

In [148]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (112.76861553, 82.41146709)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (112.76861553, 82.41146709)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (112.76861553, 82.41146709)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (113.17729236, 82.38951813)>
Using SkyOffsetFrame for Star VZ Cam 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 7.5179, 82.4115
Original RA(Deg)/Dec: 112.7686, 82.4115
New  RA(Hr)/Dec:  7.5452, 82.3895
New  RA(Deg)/Dec:  113.1773, 82.3895
Delta RA/DEC(min): -0.4087,           0.0219


### Update Properties for Star based on wiki query results above 

In [149]:
star_magnitude = 4.92;  target_alt_name = "HD 55966" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M4'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.92: 46.20 s


In [150]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR Erakis

In [151]:
from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Erakis"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star Erakis

In [152]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (325.87691482, 58.78004609)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (325.87691482, 58.78004609)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (325.87691482, 58.78004609)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (325.98126993, 58.75824632)>
Using SkyOffsetFrame for Star Erakis 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 21.7251, 58.7800
Original RA(Deg)/Dec: 325.8769, 58.7800
New  RA(Hr)/Dec:  21.7321, 58.7582
New  RA(Deg)/Dec:  325.9813, 58.7582
Delta RA/DEC(min): -0.1044,           0.0218


### Update Properties for Star based on wiki query results above 

In [153]:
star_magnitude = 4.08;  target_alt_name = "HD 206936" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.08: 22.02 s


In [154]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR 42 Her

In [155]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "42 Her"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star 42 Her

In [156]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (249.68685421, 48.92834233)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.68685421, 48.92834233)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.68685421, 48.92834233)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (249.76919818, 48.90655538)>
Using SkyOffsetFrame for Star 42 Her 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 16.6458, 48.9283
Original RA(Deg)/Dec: 249.6869, 48.9283
New  RA(Hr)/Dec:  16.6513, 48.9066
New  RA(Deg)/Dec:  249.7692, 48.9066
Delta RA/DEC(min): -0.0823,           0.0218


### Update Properties for Star based on wiki query results above 

In [157]:
star_magnitude = 4.86;  target_alt_name = "HD 150450" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'M2'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 4.86: 43.82 s


In [158]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR NEW STAR V906 Her

In [159]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "V906 Her"
md(f"### Obtain spectral details for new star {target_name}")


### Obtain spectral details for new star V906 Her

In [160]:
new_coord, orig_coord = obtain_info_for_star(target_name)

<SkyCoord (ICRS): (ra, dec) in deg
    (249.63558952, 48.8622953)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.63558952, 48.8622953)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (249.63558952, 48.8622953)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (249.71782485, 48.84050843)>
Using SkyOffsetFrame for Star V906 Her 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 16.6424, 48.8623
Original RA(Deg)/Dec: 249.6356, 48.8623
New  RA(Hr)/Dec:  16.6479, 48.8405
New  RA(Deg)/Dec:  249.7178, 48.8405
Delta RA/DEC(min): -0.0822,           0.0218


### Update Properties for Star based on wiki query results above 

In [161]:
star_magnitude = 6.60;  target_alt_name = "HD 150409" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Ma'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 6.6: 203.42 s


In [162]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR Planet Neptune

In [163]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Neptune"
md(f"### Obtain spectral details for Planet {target_name}")


### Obtain spectral details for Planet Neptune

In [164]:
#current 08/15/25 6:30 PM Pacific coordinates for Neptune
planet_ra = 0.079 #decimal deg
planet_dec = -1.344 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord, orig_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (0.079, -1.344)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0.079, -1.344)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0.079, -1.344)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (0.13313916, -1.36575702)>
Using SkyOffsetFrame for Star Neptune 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.0053, -1.3440
Original RA(Deg)/Dec: 0.0790, -1.3440
New  RA(Hr)/Dec:  0.0089, -1.3658
New  RA(Deg)/Dec:  0.1331, -1.3658
Delta RA/DEC(min): -0.0541,           0.0218


### Update Properties for Star based on wiki query results above 

In [165]:
star_magnitude = 7.8;  target_alt_name = "Neptune" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 7.8: 586.41 s


In [166]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR Planet Saturn

In [167]:
# CELESTIAL COORDINATES FOR NEW STAR

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Saturn"
md(f"### Obtain spectral details for Planet {target_name}")


### Obtain spectral details for Planet Saturn

In [168]:
#current 08/15/25 6:30 PM Pacific coordinates for Neptune
planet_ra = 1.6195 #decimal deg
planet_dec = -1.9441 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord, orig_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (1.6195, -1.9441)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (1.6195, -1.9441)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (1.6195, -1.9441)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (1.67365565, -1.96585675)>
Using SkyOffsetFrame for Star Saturn 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.1080, -1.9441
Original RA(Deg)/Dec: 1.6195, -1.9441
New  RA(Hr)/Dec:  0.1116, -1.9659
New  RA(Deg)/Dec:  1.6737, -1.9659
Delta RA/DEC(min): -0.0542,           0.0218


### Update Properties for Star based on wiki query results above 

In [169]:
star_magnitude = 0.99;  target_alt_name = "Saturn" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 0.99: 1.44 s


In [170]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# CELESTIAL COORDINATES FOR Planet Uranus

In [171]:
# CELESTIAL COORDINATES FOR Planet Uranus

from IPython.display import Markdown as md
# Instead of setting the cell to Markdown, create Markdown from withnin a code cell!
# We can just use python variable replacement syntax to make the text dynamic
target_name = "Uranus"
md(f"### Obtain spectral details for Planet {target_name}")


### Obtain spectral details for Planet Uranus

In [172]:
#current 08/15/25 6:30 PM Pacific coordinates for Uranus
planet_ra = 52.504 #decimal deg
planet_dec = 18.711 #decimal deg
planet_coord = SkyCoord(ra=planet_ra * u.deg, dec=planet_dec * u.deg, frame='icrs')
new_coord, orig_coord = obtain_adjcoord_for_planet(target_name, planet_coord)

<SkyCoord (ICRS): (ra, dec) in deg
    (52.504, 18.711)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (52.504, 18.711)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (52.504, 18.711)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (52.56113656, 18.68923372)>
Using SkyOffsetFrame for Star Uranus 
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 3.5003, 18.7110
Original RA(Deg)/Dec: 52.5040, 18.7110
New  RA(Hr)/Dec:  3.5041, 18.6892
New  RA(Deg)/Dec:  52.5611, 18.6892
Delta RA/DEC(min): -0.0571,           0.0218


### Update Properties for Star based on wiki query results above 

In [173]:
star_magnitude = 5.75;  target_alt_name = target_name # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Pl'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 5.75: 96.09 s


In [174]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# OUTPUT CONSOLIDATED CSV FILE FOR BARO

In [175]:
df.to_csv(f"{data_folder_name}/{output_csvfilename}")
print(df[["Name1*","Name2*","RA2000*","D2000*","Pmag~","Exp~","Note1","Note2","NExp~","GetRef","Temp","rahrdec","rahrdegdec","decdegdec"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

# STOP HERE ADHOC CELESTIAL COORDINATES FOR ADHOC TARGET

In [176]:
import ipywidgets as widgets
from IPython.display import display

text_input = widgets.Text(description='Enter Adhoc Target Name:')
output = widgets.Output()
current_text_value = ""

def on_text_change(change):
    global current_text_value
    current_text_value = change['new']
    with output:
        output.clear_output(wait=True)
        print(f"Current input value: {current_text_value}")

text_input.observe(on_text_change, names='value')
display(text_input, output)

# Create an float input widget for RA
ra_widget = widgets.FloatText(
    value=0.0000,
    description='Enter a RA in decimal degrees for adhoc target:',
    disabled=False
)

# Create an float input widget for DEC
dec_widget = widgets.FloatText(
    value=0.0000,
    description='Enter a DEC in decimal degrees for adhoc target:',
    disabled=False
)

display(ra_widget)
display(dec_widget)

# You can access the value in another cell or later in the same cell's execution
# with `num_widget.value`
# Example:
# my_number = num_widget.value
# print(f"The number entered is: {my_number}")


Text(value='', description='Enter Adhoc Target Name:')

Output()

FloatText(value=0.0, description='Enter a RA in decimal degrees for adhoc target:')

FloatText(value=0.0, description='Enter a DEC in decimal degrees for adhoc target:')

In [177]:
my_tgt = text_input.value
my_ra = ra_widget.value
my_dec = dec_widget.value
print(f"The values entered are: {my_tgt} & {my_ra} & {my_dec}")

The values entered are:  & 0.0 & 0.0


In [178]:
new_coord = obtain_info_for_adhoc_star(my_tgt, my_ra, my_dec)

<SkyCoord (ICRS): (ra, dec) in deg
    (0., 0.)>

sky_PA: 291.9 deg

offset_dist: -3.5 arcmin

dx: 3.247426888646221 arcmin dy: -1.3054572390153285 arcmin

offset_frame = <SkyOffsetICRS Frame (rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0., 0.)>)>

offset_coord = <SkyCoord (SkyOffsetICRS: rotation=0.0 deg, origin=<ICRS Coordinate: (ra, dec) in deg
    (0., 0.)>): (lon, lat) in deg
    (0.05412378, -0.02175762)>

new_coord_ = <SkyCoord (ICRS): (ra, dec) in deg
    (0.05412378, -0.02175762)>
Using SkyOffsetFrame for Star  
User inputs: offset_arcmin = -3.5, camera_rotation_deg = -21.9

Original RA(Hr)/Dec: 0.0000, 0.0000
Original RA(Deg)/Dec: 0.0000, 0.0000
New  RA(Hr)/Dec:  0.0036, -0.0218
New  RA(Deg)/Dec:  0.0541, -0.0218
Delta RA/DEC(min): -0.0541,           0.0218


### Update Properties for Star based on wiki query results above 

In [179]:
star_magnitude = 12.9;  target_alt_name = "Quasar_3C273" # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 12.9: 52772.22 s


In [180]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     

In [181]:
star_magnitude = 12.9;  target_alt_name = my_tgt # FIX THIS LINE AND BELOW BASED ON SEARCH
star_type = 'Qu'
exposure = compute_exposure_time(star_magnitude)
print(f"Required exposure for mag {star_magnitude}: {exposure:.2f} s")

Required exposure for mag 12.9: 52772.22 s


In [182]:
df.loc[ridx]=[f'{target_name}_{target_alt_name}_Typ_{star_type}',f'{target_alt_name}',f'{new_coord.ra.deg:.4f}',f'{new_coord.dec.deg:.4f}',
           f'{star_magnitude}',f'{exposure:.2f}','NA','NA','1','0','',f'{orig_coord.ra.hour:.4f}',f'{orig_coord.ra.deg:.4f}',f'{orig_coord.dec.deg:.4f}']
ridx += 1
df['Name1*'] = df['Name1*'].str.replace(' ', '_'); print(df[['Name1*','Name2*',"RA2000*","D2000*","Pmag~","Exp~"]])

                             Name1*        Name2*   RA2000*    D2000*  Pmag~  \
0         3C273_Quasar_3C273_Typ_Qu  Quasar_3C273  187.3321    2.0306   12.9   
1          HD_45351_HD108959_Typ_A3      HD108959   97.1863   34.4708   7.73   
2           HR_5422_HR_5422_Typ_A0V       HR 5422  217.5206   31.7694   6.05   
3         Alphecca_Alphecca_Typ_A0V      Alphecca  233.7325   26.6929   2.22   
4   Rasalhague_Rasalhague_Typ_AVIII    Rasalhague  263.7891   12.5383   2.08   
5           Altair_HD_187642_Typ_A7     HD 187642  297.7506    8.8466   0.76   
6             Vega_HD_172167_Typ_A0     HD 172167  279.3041   38.7619  0.026   
7             Dubhe_HD_95689_Typ_K0      HD 95689  166.0462   61.7292   1.79   
8           Scheat_HD_217906_Typ_M2     HD 217906  346.0049   28.0610   2.42   
9            Mizar_HD_116656_Typ_A2     HD 116656  201.0756   54.9036   2.04   
10           Alcor_HD_116657_Typ_MK     HD 116657  201.4007   54.9662   3.88   
11           R_Lyr_HD_175865_Typ_M5     